# Multi-tool comparison for the HSV-2 candidate set

This notebook loads the completed v0.3 run and uses the v0.4 comparison API. The 32 real-run candidates are computational candidates for which Cas-OFFinder reported no predicted human hit through three mismatches under the configured SpCas9/GRCh38.p14 model. That is not proof of safety, efficiency, or experimental validation. A bundled CSV fixture is used when local real-run outputs are unavailable.

In [ ]:
from pathlib import Path
import pandas as pd
import viral_safe_target as vst
from viral_safe_target.notebook_helpers import find_project_root
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

root = find_project_root()
real_candidates = root / 'reports/hsv2_pilot/candidates_ranked_post_human.csv'
if real_candidates.exists():
    run = vst.load_run(root / 'reports/hsv2_pilot')
    candidates = run.candidates.query('human_total_predicted_hits == 0').copy()
    assert len(candidates) == 32
else:
    candidates = pd.read_csv(root / 'examples/demo_output/candidates.csv')
    candidates['pre_human_score'] = candidates['demo_score']
    candidates['post_human_score'] = candidates['demo_score']
    candidates['human_total_predicted_hits'] = candidates['host_matches_le_3_mismatches']
    print('Fixture mode: the local 32-candidate run is unavailable.')
candidates['gene_name'].value_counts()

## What each tool contributes

ViralSafeTarget prioritizes conservation and transparent sequence features. Cas-OFFinder enumerates reference-genome similarities. CRISPRitz can add bulges and variant-aware searches. CRISPOR, CHOPCHOP, and GuideScan2 are imported from documented researcher exports. These raw metrics measure different things, so the comparison uses within-tool ranks and percentiles rather than averaging raw values.

In [ ]:
baseline = vst.candidate_metrics_as_tool_results(candidates)
comparison = vst.compare_tools(
    candidates, [baseline],
    expected_tools=[
        'viral_safe_target_pre_human', 'viral_safe_target_post_human',
        'cas-offinder', 'crispritz', 'crispor', 'chopchop', 'guidescan2',
    ],
)
comparison.candidate_tool_matrix.head()

## Leading candidates, missing tools, and disagreement

The matrix preserves `NaN` for unavailable tools. `tools_missing`, rank variance, and `disagreement_score` keep partial coverage visible. Consensus is prioritization, not evidence that the majority model is biologically correct.

In [ ]:
display(comparison.consensus_candidates.head(10))
display(comparison.disagreement_report.head(10))
display(comparison.tool_coverage)

## Registering a custom scorer

A researcher can implement an object with `name`, `version`, and `score(candidates)`. The transparent example below demonstrates the API contract; it is not a validated biological model.

In [ ]:
scored = vst.ExampleRuleScorer().score(candidates)
scored.head()

## Future CRISPResso2 results

CRISPResso2 represents measured sequencing results from an existing experiment. Its imports remain under `experimental/` and are not automatically aggregated with prediction scores. Future analysis should display prediction beside measurement while retaining provenance and uncertainty.